In [ ]:
import sys
import os
import re
import joblib
import numpy as np
import gensim
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from slovene_pipeline.word2vec_api import Word2VecAPI, maybe_load_emoji2vec
from slovene_pipeline.features import build_tfidf, build_w2v_mean, combine_features
from irony_translation.LLMSarcasmTranslator import LLMSarcasmTranslator
from irony_translation.SarcasmTranslator import T5SarcasmTranslator

In [ ]:
MODEL_PATH = os.path.join(project_root, 'slovene_pipeline', 'subtaskA_notebooks', 'finalized_model_og.sav')

if os.path.exists(MODEL_PATH):
    irony_classifier = joblib.load(MODEL_PATH)
    if isinstance(irony_classifier, dict) and "vectorizer" in irony_classifier:
        vectorizer = irony_classifier["vectorizer"]
        clf = irony_classifier["classifier"]
    else:
        clf = irony_classifier
        vectorizer = None 
else:
    print(f"Model not found at {MODEL_PATH}")

kv = gensim.models.fasttext.load_facebook_model('../w2v-slo/all-token-prelim.ft.sg.bin').wv
w2v = Word2VecAPI(kv)
emoji_model = maybe_load_emoji2vec(
    os.path.join(project_root, 'emoji2vec-master', 'pre-trained', 'emoji2vec.bin'),
    binary=True
)

# Option 1: LLM Translator (Groq/OpenAI base)
API_KEY = "api_key_here" 
llm_translator = LLMSarcasmTranslator(api_key=API_KEY)

# Option 2: Local HuggingFace Translator (e.g., mT5 trained on dataset)
local_translator = T5SarcasmTranslator(model_name="csebuetnlp/mT5_multilingual_XLSum")
# Uncomment next line to load local weights if trained locally
local_translator.load_model(os.path.join(project_root, "irony_translation", "from_medium_dataset", "nllb_200_distilled_600M"))

# Choose which one to pass to the pipeline
translator = llm_translator 
#translator = local_translator

In [ ]:
def extract_enhanced_features(tweet_text):
    """
    Extract enhanced NLP features for irony/sarcasm detection.
    Returns exactly 58 features (ORIGINAL ENGLISH BASELINE) without pragmatic priors.
    """
    from ekphrasis.utils.nlp import polarity
    import re

    words = tweet_text.split()
    words_lower = [w.lower() for w in words]

    # SLOVENE SENTIMENT WORDS
    positive_words = ['rad', 'obožujem', 'lepo', 'super', 'odličko', 'hvala', 'rabi', 'rabu',
                      'super', 'great', 'love', 'perfect', 'amazing', 'wonderful', 'fantastic']
    negative_words = ['zaprta', 'zaprto', 'zaprte', 'slabo', 'čudno', 'groza', 'strašno',
                      'problematično', 'dead', 'horrible', 'ugly', 'terrible', 'bad']

    # 1) Positive-negative contradiction
    positive_count = sum(1 for w in words_lower if any(w.startswith(p) or p in w for p in positive_words))
    negative_count = sum(1 for w in words_lower if any(w.startswith(n) or n in w for n in negative_words))
    sarcasm_contrast = 1 if (positive_count > 0 and negative_count > 0) else 0

    # 2) Left-right intensity/contrast
    left_half = words[:len(words)//2]
    right_half = words[len(words)//2:]
    left_word_lens = [len(w) for w in left_half]
    right_word_lens = [len(w) for w in right_half]

    left_intensity = 1 if (sum(left_word_lens) / max(len(left_half), 1)) < 4 else 0
    right_intensity = 1 if (sum(right_word_lens) / max(len(right_half), 1)) < 4 else 0
    polarity_diff = 1 if len(left_half) > 0 and len(right_half) > 0 else 0

    contrast = 0
    try:
        if len(left_half) > 0 and len(right_half) > 0:
            left_text = ' '.join(left_half)
            right_text = ' '.join(right_half)
            left_pol = polarity(left_text)
            right_pol = polarity(right_text)
            if (left_pol and right_pol and len(left_pol) > 1 and len(right_pol) > 1):
                contrast = 1 if abs(left_pol[0] - right_pol[0]) > 0.5 else 0
    except:
        pass

    # 3) Punctuation/linguistic markers
    exclamation_count = tweet_text.count('!')
    question_count = tweet_text.count('?')
    ellipsis_count = tweet_text.count('...')
    ellipsis_signal = 1 if ellipsis_count > 0 else 0
    excessive_punct = 1 if (exclamation_count > 2 or question_count > 2) else 0

    elongated_words = len([w for w in words if re.search(r'(.)\1{2,}', w)])
    elongation_score = min(elongated_words / max(len(words), 1), 1.0)

    slovene_negations = ['ne', 'nema', 'nimam', 'nimajo', 'nič', 'nikoli', 'nobeden', 'brez']
    negation_count = sum(1 for word in words_lower if any(word.startswith(neg) or neg in word for neg in slovene_negations))
    negation_score = min(negation_count / max(len(words), 1), 1.0)
    
    # EXACT ORIGINAL 58-DIMENSION BUILDER (NO OVERWRITING, NO APPENDING)
    core = [left_intensity, right_intensity, polarity_diff, contrast]
    
    aux_54 = np.zeros(54, dtype=float)
    aux_54[0] = exclamation_count / 5
    aux_54[1] = question_count / 5
    aux_54[2] = ellipsis_count / 3
    aux_54[3] = excessive_punct
    aux_54[4] = elongation_score
    aux_54[5] = negation_score
    aux_54[6] = sarcasm_contrast
    aux_54[7] = ellipsis_signal
    aux_54[8] = positive_count / max(len(words), 1)
    aux_54[9] = negative_count / max(len(words), 1)

    # Note: In OG base features, we strictly leave slots 52 & 53 UNMODIFIED
    # No emoji sentiment prior
    # No hashtag sentiment prior
    # No pragmatic heuristics (no +2 dimensions at the end)

    all_feats = core + aux_54.tolist() # Length: 4 + 54 = 58
    return all_feats

In [ ]:
def is_ironic(sentence: str, clf_model=None, w2v_model=None, emoji_model=None):
    if clf_model is None:
        return False
    
    try:
        # combining all features (from words2vec, emoji2vec and extract_enhanced_features())
        w2v_feat = build_w2v_mean([sentence], w2v_model, emoji_model)[0]
        enhanced_feat = np.array(extract_enhanced_features(sentence))
        x = np.hstack([w2v_feat, enhanced_feat]).reshape(1, -1)
        
        preds = clf_model.predict(x)
        return bool(preds[0])
    except Exception as e:
        print(f"Error during sentence prediction: {e}")
        return False

In [ ]:
def translate_ironic_sentence(sentence: str, translator_model) -> str:
    try:
        if hasattr(translator_model, 'zero_shot'):
            # Used by LLMSarcasmTranslator
            non_ironic = translator_model.zero_shot(sentence)
        elif hasattr(translator_model, 'generate'):
            # Used by T5SarcasmTranslator
            non_ironic = translator_model.generate(sentence)
        else:
            return sentence
            
        return non_ironic.strip()
    except Exception as e:
        print(f"Translation failed: {e}")
        return sentence

In [ ]:
def process_text_pipeline(input_text: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None, translator_model=None):
    is_ironic_text = is_ironic(
        input_text, 
        clf_model=clf_model, 
        w2v_model=w2v_model, 
        emoji_model=emoji_model
    )
    
    final_text = input_text
    transformed = False
    
    if is_ironic_text and translator_model is not None:
        final_text = translate_ironic_sentence(input_text, translator_model)
        transformed = True
        
    result = {
        "original_text": input_text,
        "final_text": final_text,
        "is_ironic": is_ironic_text,
        "transformed": transformed
    }
    
    return result

In [ ]:
print("clf:", clf)
print("vectorizer:", vectorizer)
print("w2v:", w2v)
print("emoji_model:", emoji_model)
print("translator:", translator)

Example usage:

In [ ]:
sample_texts = [
    "Oh, kako obožujem dve uri čakanja v koloni. Najboljši način za začetek jutra! 🚗🕰️ #zastoji #ljubljana #sreča",
    "To, da mi telefon crkne pri 2 %, ravno ko potrebujem navigacijo, je brez dvoma moja najljubša funkcija. 📱🔋 #tehnologija #popolno",
    "Uau, še en deževen vikend. Saj sem bil res že pošteno utrujen od preveč sonca letos. 🌧️☔ #vreme #slovenija #komajčakam",
    "Danes sem imel čudovit pohod na Šmarno goro! Razgled je na koncu vedno poplačan. 🏔️☀️ #šmarnagora #hribi #narava"
]
CLF = globals().get('clf', None)
VEC = globals().get('vectorizer', None)
W2V = globals().get('w2v', None)
EMOJI = globals().get('emoji_model', None)
TRANS = globals().get('translator', None)
for text in sample_texts:

    
    if CLF is None:
        def mock_is_ironic(sentence, *args, **kwargs):
            return "odličen" in sentence.lower() and "dežuje" in sentence.lower() or "seveda" in sentence.lower()
        
        original_is_ironic = is_ironic
        is_ironic = mock_is_ironic
    
    output = process_text_pipeline(text, clf_model=CLF, vectorizer_model=VEC, w2v_model=W2V, emoji_model=EMOJI, translator_model=TRANS)
    
    import json
    print(json.dumps(output, indent=2, ensure_ascii=False))
    
    if CLF is None:
         is_ironic = original_is_ironic

In [ ]:
import pandas as pd
# IMPORTANT tu je dejanski experiment
def run_experiment_on_file(input_csv_path: str, output_csv_path: str, ironic_csv_path: str = None):
    df = pd.read_csv(input_csv_path)
    
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    results = []
    ironic_for_rerun = []
    
    for index, row in df.iterrows():
        text = str(row['text'])
        true_label = row.get('true_label', '')
        
        output = process_text_pipeline(
            text, 
            clf_model=CLF, 
            vectorizer_model=VEC, 
            w2v_model=W2V, 
            emoji_model=EMOJI, 
            translator_model=TRANS
        )
        
        results.append({
            'input_text': text,
            'true_label': true_label,
            'predicted_ironic': output.get('is_ironic', False),
            'output_text': output.get('final_text', text),
        })
        
        if output.get('is_ironic', False):
            ironic_for_rerun.append({
                'text': output.get('final_text', text),  # translated output becomes the new input
                'true_label': False,
            })
        
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Eksperiment zaključen. Podatki shranjeni v: {output_csv_path}")

    if ironic_for_rerun:
        ironic_path = ironic_csv_path or output_csv_path.replace('.csv', '_ironic_rerun.csv')
        ironic_df = pd.DataFrame(ironic_for_rerun)
        ironic_df.to_csv(ironic_path, index=False, encoding='utf-8-sig')
        print(f"Ironični primeri za ponovni zagon shranjeni v: {ironic_path} ({len(ironic_for_rerun)} vrstic)")
    else:
        print("Ni ironičnih primerov za ponovni zagon.")

# First run - full dataset, detects & translates ironic sentences
run_experiment_on_file("experiment_input.csv", "rezultati_eksperimenta_normal.csv", ironic_csv_path="za_ponovni_zagon.csv")

# Second run - only the translated sentences, check if they're now detected as non-ironic
run_experiment_on_file("za_ponovni_zagon.csv", "rezultati_ponovni_zagon.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

def plot_confusion_matrix(csv_path: str):
    df = pd.read_csv(csv_path)

    y_true = df['true_label'].astype(int)

    y_pred = df['predicted_ironic'].map({
        True: 1,
        False: 0,
        'True': 1,
        'False': 0
    }).astype(int)

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Ne ironično (0)', 'Ironično (1)']
    )

    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    # Custom axis labels
    ax.set_xlabel('Napoved modela')
    ax.set_ylabel('Pravilna oznaka')

    ax.set_title(
        'Matrika zmede - klasifikacija transformacije na podlagi LLM',
        fontsize=13,
        pad=12
    )

    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=['Ne ironično', 'Ironično']
    ))

# usage
plot_confusion_matrix("rezultati_ponovni_zagon.csv")